In [1]:
import boto3
import os
import json
import random
from datetime import datetime, timedelta, timezone
from botocore.exceptions import ClientError

# --- CONFIGURACIÓN GLOBAL ---
# Usaremos estos nombres para los ejemplos, cámbialos si es necesario
BUCKET_PRINCIPAL = 'alan670265371833'

In [5]:
# =================================================================
# PROGRAMA 1: LISTAR BUCKETS
# =================================================================
print("--- PROGRAMA 1: LISTAR BUCKETS ---")
s3_client = boto3.client('s3')
response = s3_client.list_buckets()
print("Tus cubetas disponibles:")
for bucket in response['Buckets']:
    print(f"- {bucket['Name']}")

--- PROGRAMA 1: LISTAR BUCKETS ---
Tus cubetas disponibles:
- alan670265371833


In [ ]:
# =================================================================
# PROGRAMA 2: OPERACIONES BÁSICAS S3 (CORREGIDO)
# =================================================================
print("\n--- PROGRAMA 2: OPERACIONES S3 ---")

def asegurar_bucket(nombre_bucket):
    region_actual = s3_client.meta.region_name
    try:
        # Verificar si el bucket existe
        s3_client.head_bucket(Bucket=nombre_bucket)
        print(f"El bucket '{nombre_bucket}' ya existe.")
        return True
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == '404':
            print(f"El bucket '{nombre_bucket}' no existe. Creándolo en {region_actual}...")
            try:
                if region_actual == 'us-east-1':
                    s3_client.create_bucket(Bucket=nombre_bucket)
                else:
                    s3_client.create_bucket(
                        Bucket=nombre_bucket,
                        CreateBucketConfiguration={'LocationConstraint': region_actual}
                    )
                print(f"Bucket '{nombre_bucket}' creado exitosamente.")
                return True
            except Exception as ex:
                print(f"Error crítico al crear el bucket: {ex}")
                return False
        else:
            print(f"Error de permisos o configuración: {e}")
            return False

def s3_operations():
    bucket_name = BUCKET_PRINCIPAL 
    nombre_archivo_subida = 'archivo_local.txt'
    
    if not asegurar_bucket(bucket_name):
        print("No se pueden realizar operaciones porque el bucket no está disponible.")
        return

    # Crear archivo local de prueba
    with open(nombre_archivo_subida, 'w') as f:
        f.write("Contenido de prueba para S3 generado desde el Notebook.")

    try:
        # Subir archivo
        print(f"Intentando subir {nombre_archivo_subida}...")
        s3_client.upload_file(nombre_archivo_subida, bucket_name, 'remoto.txt')
        print(f"Subida exitosa a {bucket_name}.")

        # Listar
        print("\nArchivos actualmente en el bucket:")
        objs = s3_client.list_objects_v2(Bucket=bucket_name)
        if 'Contents' in objs:
            for obj in objs['Contents']:
                print(f" - {obj['Key']}")
        else:
            print(" El bucket está vacío.")

        # Descargar
        s3_client.download_file(bucket_name, 'remoto.txt', 'descargado_de_s3.txt')
        print("\nDescarga exitosa. Archivo guardado como 'descargado_de_s3.txt'")

    except Exception as e:
        print(f"Error durante la ejecución en S3: {e}")

s3_operations()


--- PROGRAMA 2: OPERACIONES S3 ---
El bucket 'alan670265371833' no existe. Creándolo en us-west-2...
Bucket 'alan670265371833' creado exitosamente.
Intentando subir archivo_local.txt...
Subida exitosa a alan670265371833.

Archivos actualmente en el bucket:
 - remoto.txt

Descarga exitosa. Archivo guardado como 'descargado_de_s3.txt'


In [6]:
# =================================================================
# PROGRAMA 3: BACKUP AUTOMÁTICO CON GENERACIÓN DE ARCHIVOS
# =================================================================
print("\n--- PROGRAMA 3: BACKUP DE CARPETA ---")

import os
import boto3
from datetime import datetime

# Usamos el recurso de S3 y la variable global del bucket
s3_res = boto3.resource('s3')
ruta_local_backup = './mi_carpeta_backups'

def preparar_archivos_locales(ruta):
    """Crea una carpeta y genera 5 archivos de texto con info random"""
    if not os.path.exists(ruta):
        os.makedirs(ruta)
        print(f"Carpeta '{ruta}' creada.")
    
    print("Generando 5 archivos de prueba...")
    for i in range(1, 6):
        nombre_archivo = f"archivo_prueba_{i}.txt"
        ruta_completa = os.path.join(ruta, nombre_archivo)
        
        # Contenido random sencillo
        contenido = f"Este es el archivo número {i}\nGenerado el: {datetime.now()}\nID Aleatorio: {os.urandom(4).hex()}"
        
        with open(ruta_completa, "w") as f:
            f.write(contenido)
    print("Archivos listos para el backup.")

def backup_a_s3():
    # 1. Preparar la carpeta y los archivos localmente
    preparar_archivos_locales(ruta_local_backup)
    
    # 2. Configurar estructura de S3
    fecha_hoy = datetime.now().strftime('%Y-%m-%d')
    bucket = s3_res.Bucket(BUCKET_PRINCIPAL)
    
    print(f"\nIniciando subida al bucket: {BUCKET_PRINCIPAL}...")
    
    # 3. Escanear carpeta y subir
    conteo = 0
    for raiz, dirs, archivos in os.walk(ruta_local_backup):
        for nombre_archivo in archivos:
            ruta_completa = os.path.join(raiz, nombre_archivo)
            
            # Estructura requerida: backups/AAAA-MM-DD/archivo.txt
            clave_s3 = f"backups/{fecha_hoy}/{nombre_archivo}"
            
            try:
                bucket.upload_file(ruta_completa, clave_s3)
                print(f" ✅ Respaldado con éxito: {clave_s3}")
                conteo += 1
            except Exception as e:
                print(f" ❌ Error al subir {nombre_archivo}: {e}")

    print(f"\nBackup finalizado. Se subieron {conteo} archivos.")

# Ejecutamos el programa
backup_a_s3()


--- PROGRAMA 3: BACKUP DE CARPETA ---
Carpeta './mi_carpeta_backups' creada.
Generando 5 archivos de prueba...
Archivos listos para el backup.

Iniciando subida al bucket: alan670265371833...
 ✅ Respaldado con éxito: backups/2026-04-09/archivo_prueba_3.txt
 ✅ Respaldado con éxito: backups/2026-04-09/archivo_prueba_4.txt
 ✅ Respaldado con éxito: backups/2026-04-09/archivo_prueba_2.txt
 ✅ Respaldado con éxito: backups/2026-04-09/archivo_prueba_5.txt
 ✅ Respaldado con éxito: backups/2026-04-09/archivo_prueba_1.txt

Backup finalizado. Se subieron 5 archivos.


In [9]:
# =================================================================
# PROGRAMA 4: GESTIÓN INTERACTIVA DE EC2 (CORREGIDO)
# =================================================================
print("\n--- PROGRAMA 4: GESTIÓN DE EC2 ---")
ec2_res = boto3.resource('ec2')

def gestionar_ec2_interactivo():
    try:
        # 1. Obtener todas las instancias
        instancias_existentes = list(ec2_res.instances.all())
        
        print(f"Buscando instancias disponibles...")
        
        if not instancias_existentes:
            print("No se encontraron instancias en tu cuenta.")
            crear_nueva = "s" # Si no hay nada, forzamos la creación o preguntamos
        else:
            print("\nInstancias actuales detectadas:")
            print(f"{'ID Instancia':<20} | {'Estado':<12} | {'Nombre'}")
            print("-" * 50)
            for inst in instancias_existentes:
                nombre = next((tag['Value'] for tag in (inst.tags or []) if tag['Key'] == 'Name'), "Sin nombre")
                print(f"{inst.id:<20} | {inst.state['Name']:<12} | {nombre}")
            
            # Preguntar si desea crear una nueva a pesar de que ya existen
            crear_nueva = input("\n¿Deseas crear una nueva instancia? (s/n): ").lower().strip()

        # 2. Lógica de creación
        if crear_nueva == 's':
            nombre_instancia = input("Ingresa el nombre para tu nueva instancia (Tag Name): ")
            
            print(f"🚀 Solicitando creación de instancia t3.micro...")
            
            # Nota: Asegúrate de que esta AMI sea válida en tu región (esta es de us-east-1)
            nueva_inst = ec2_res.create_instances(
                ImageId='ami-0a914de4dc1f18727', 
                MinCount=1,
                MaxCount=1,
                InstanceType='t3.micro',
                TagSpecifications=[{
                    'ResourceType': 'instance',
                    'Tags': [{'Key': 'Name', 'Value': nombre_instancia}]
                }]
            )
            
            print(f" Éxito - Instancia '{nombre_instancia}' creada con ID: {nueva_inst[0].id}")
        else:
            print("Operación de creación cancelada. Mostrando inventario actual únicamente.")

    except ClientError as e:
        print(f" Error de AWS: {e.response['Error']['Message']}")
    except Exception as e:
        print(f" Ocurrió un error inesperado: {e}")

# Ejecución del programa 4
gestionar_ec2_interactivo()


--- PROGRAMA 4: GESTIÓN DE EC2 ---
Buscando instancias disponibles...

Instancias actuales detectadas:
ID Instancia         | Estado       | Nombre
--------------------------------------------------
i-0c07e63ec403886fe  | running      | Cafe_Prueba
i-06ae03c63db22abe9  | terminated   | InstanciaNotebook
Operación de creación cancelada. Mostrando inventario actual únicamente.


In [ ]:
# =================================================================
# PROGRAMA 5: LIMPIEZA DE BUCKET (ARCHIVOS ANTIGUOS)
# =================================================================
print("\n--- PROGRAMA 5: LIMPIEZA DE BUCKET ---")
def limpiar_bucket():
    bucket = s3_res.Bucket(BUCKET_PRINCIPAL)
    limite = datetime.now(timezone.utc) - timedelta(days=30)
    for objeto in bucket.objects.all():
        if objeto.last_modified < limite:
            print(f"Eliminando {objeto.key}...")
            objeto.delete()
    print("Limpieza finalizada.")

limpiar_bucket()

In [22]:
# =================================================================
# PROGRAMA 6: CLOUDWATCH Y SNS (ALERTAS)
# =================================================================
print("\n--- PROGRAMA 6: ALERTAS CLOUDWATCH ---")
cw_client = boto3.client('cloudwatch')
sns_client = boto3.client('sns')

def sistema_alertas(email):
    try:
        topic = sns_client.create_topic(Name='AlertasNotebook')
        arn = topic['TopicArn']
        sns_client.subscribe(TopicArn=arn, Protocol='email', Endpoint=email)
        
        cw_client.put_metric_alarm(
            AlarmName='Alarma_Notebook_Errores',
            ComparisonOperator='GreaterThanThreshold',
            EvaluationPeriods=1,
            MetricName='ErroresCriticos',
            Namespace='Notebook/Monitoreo',
            Period=60,
            Statistic='Sum',
            Threshold=5.0,
            AlarmActions=[arn]
        )
        
        # Simular envío de métrica
        valor = random.randint(1, 10)
        print("Valor aleatorio" + str(valor))  
        cw_client.put_metric_data(
            Namespace='Notebook/Monitoreo',
            MetricData=[{'MetricName': 'ErroresCriticos', 'Value': valor, 'Unit': 'Count'}]
        )
        print(f"Métrica enviada: {valor}. Alerta configurada para: {email}")
    except Exception as e:
        print(f"Error en CloudWatch: {e}")

sistema_alertas("alangoku@hotmail.com")
#sistema_alertas("tu_correo@ejemplo.com")


--- PROGRAMA 6: ALERTAS CLOUDWATCH ---
Valor aleatorio5
Métrica enviada: 5. Alerta configurada para: alangoku@hotmail.com


In [ ]:
# =================================================================
# PROGRAMA 7: CREAR USUARIO IAM - Este no funciona por los permisos de la cuenta
# =================================================================
print("\n--- PROGRAMA 7: CREACIÓN USUARIO IAM ---")
iam_client = boto3.client('iam')

def crear_usuario(nombre):
    try:
        iam_client.create_user(UserName=nombre)
        iam_client.attach_user_policy(UserName=nombre, PolicyArn='arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess')
        keys = iam_client.create_access_key(UserName=nombre)
        print(f"Usuario {nombre} creado con éxito.")
        return {"AccessKey": keys['AccessKey']['AccessKeyId'], "Secret": "******"}
    except iam_client.exceptions.EntityAlreadyExistsException:
        return "El usuario ya existe."

print(crear_usuario('usuario_prueba_notebook'))

In [24]:
# =================================================================
# PROGRAMA 8: INVENTARIO GENERAL A JSON
# =================================================================
print("\n--- PROGRAMA 8: INVENTARIO GENERAL ---")
def generar_inventario():
    inventario = {"ec2": [], "s3": [], "lambda": []}
    
    # EC2 IDs
    ec2_c = boto3.client('ec2')
    res = ec2_c.describe_instances()
    for r in res['Reservations']:
        for i in r['Instances']: inventario["ec2"].append(i['InstanceId'])
    
    # S3 Buckets
    inventario["s3"] = [b['Name'] for b in s3_client.list_buckets()['Buckets']]
    
    # Lambda Functions
    lam = boto3.client('lambda')
    for f in lam.list_functions()['Functions']: inventario["lambda"].append(f['FunctionName'])
    
    with open('aws_inventory.json', 'w') as f:
        json.dump(inventario, f, indent=4)
    print("Inventario guardado en 'aws_inventory.json'")

generar_inventario()


--- PROGRAMA 8: INVENTARIO GENERAL ---
Inventario guardado en 'aws_inventory.json'


In [25]:
# =================================================================
# PROGRAMA 8: PIPELINE SIMPLE S3 (PROCESAMIENTO)
# =================================================================
print("\n--- PROGRAMA 8: PIPELINE S3 ---")
import boto3

s3 = boto3.client('s3')
nombre_bucket = 'alan670265371833'
archivo_entrada = 'datos.txt'
archivo_salida = 'resultado.txt'

with open(archivo_entrada, 'w') as f:
    f.write("Línea 1: Hola Mundo\nLínea 2: Aprendiendo Boto3\nLínea 3: AWS es genial")

s3.upload_file(archivo_entrada, nombre_bucket, 'entrada/datos.txt')
print("Archivo original subido.")

s3.download_file(nombre_bucket, 'entrada/datos.txt', 'temporal.txt')

with open('temporal.txt', 'r') as f:
    lineas = f.readlines()
    total = len(lineas)

resultado_texto = f"El archivo procesado tiene un total de: {total} líneas."
with open(archivo_salida, 'w') as f:
    f.write(resultado_texto)

s3.upload_file(archivo_salida, nombre_bucket, 'resultados/reporte_lineas.txt')

print(f"Procesamiento terminado. Resultado guardado en S3 como 'resultados/reporte_lineas.txt'")


--- PROGRAMA 8: PIPELINE S3 ---
Archivo original subido.
Procesamiento terminado. Resultado guardado en S3 como 'resultados/reporte_lineas.txt'


In [26]:
# =================================================================
# PROGRAMA 9: SNAPSHOT DE VOLÚMENES EBS
# =================================================================
print("\n--- PROGRAMA 9: SNAPSHOT AUTOMÁTICO ---")
import boto3

ec2 = boto3.client('ec2')

# 1. Buscar todos los volúmenes en la región
print("Buscando volúmenes disponibles...")
respuesta = ec2.describe_volumes()

for volumen in respuesta['Volumes']:
    vol_id = volumen['VolumeId']
    estado = volumen['State']
    print(f"Encontrado volumen: {vol_id} en estado: {estado}")
    
    # 2. Crear un snapshot (respaldo) de cada volumen encontrado
    print(f"Creando snapshot para {vol_id}...")
    snap = ec2.create_snapshot(
        VolumeId=vol_id,
        Description=f'Respaldo creado por mi script de aprendizaje - {vol_id}'
    )
    print(f"Snapshot creado con ID: {snap['SnapshotId']}")

print("Proceso de snapshots finalizado.")


--- PROGRAMA 9: SNAPSHOT AUTOMÁTICO ---
Buscando volúmenes disponibles...
Encontrado volumen: vol-0918140fc02cfde5b en estado: in-use
Creando snapshot para vol-0918140fc02cfde5b...
Snapshot creado con ID: snap-0ef48c1af4e817e8f
Proceso de snapshots finalizado.


In [27]:
# =================================================================
# PROGRAMA 10: GENERADOR DE REPORTES
# =================================================================
print("\n--- PROGRAMA 10: GENERADOR DE REPORTES ---")
import boto3

s3 = boto3.client('s3')
ec2 = boto3.client('ec2')
nombre_bucket = 'alan670265371833'

print("Obteniendo datos para el reporte...")

# 1. Recolectar info simple (Buckets e Instancias)
buckets = s3.list_buckets()['Buckets']
instancias = ec2.describe_instances()['Reservations']

# 2. Escribir el reporte en un archivo local
nombre_reporte = "reporte_aws_simple.txt"
with open(nombre_reporte, "w") as f:
    f.write("=== REPORTE SIMPLIFICADO DE MI CUENTA AWS ===\n")
    f.write(f"Fecha: {datetime.now()}\n\n")
    
    f.write(f"CANTIDAD DE BUCKETS S3: {len(buckets)}\n")
    for b in buckets:
        f.write(f"- {b['Name']}\n")
        
    f.write(f"\nCANTIDAD DE RESERVAS EC2: {len(instancias)}\n")

# 3. Subir el reporte a S3
s3.upload_file(nombre_reporte, nombre_bucket, 'reportes/ultimo_reporte.txt')

print(f"Reporte generado y subido a S3 como 'reportes/ultimo_reporte.txt'")


--- PROGRAMA 10: GENERADOR DE REPORTES ---
Obteniendo datos para el reporte...
Reporte generado y subido a S3 como 'reportes/ultimo_reporte.txt'
